# Join page MEI files into one score

**Workflow 1 — handle MEI files.** When a piece is encoded as one MEI file per source page, this notebook joins those pages into a single `*_full.mei` score. The original page files are not changed.

The example uses ten Buxtehude pages in this repository (`test_corpus/buxtehude_pages/`). Writes stay off until you set `RUN_COMBINE = True`. The combined file goes to `converted_mei/combine_tutorial/` (gitignored).

**What you do**

1. Point `MEI_INPUTS` at a folder of pages, or list the files in score order.
2. Review the plan (nothing is written yet).
3. Set `RUN_COMBINE = True` and run the join cell.

CAMAT copies the pages first so originals stay unchanged, makes `xml:id` values unique across pages, strips leftover MusicXML `@ppq` / `@accid.ges` on those copies, concatenates the pages, numbers measures `1…n` through the piece, and keeps staff names from the first page at the start of the score.

Companion guide: [Handling MEI files](../docs/guides/edition-building.md). After joining, check the result in [`mei_check_report.ipynb`](mei_check_report.ipynb) and inspect it in [`mei_facsimile_viewer.ipynb`](mei_facsimile_viewer.ipynb). For extra flags (staff-name overrides, annotation export, combine-then-check in one run), use [`mei_consistency_checks.ipynb`](mei_consistency_checks.ipynb).

In [ ]:
# Cloud Jupyter: install CAMAT and copy notebooks/test_corpus if they are not already here.
try:
    import setup_camat
except ModuleNotFoundError:
    pass
try:
    from camat.notebook_workspace import prepare_notebook
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "camat"])
    from camat.notebook_workspace import prepare_notebook

prepare_notebook()

# You can leave this cell unchanged.
try:
    from camat import (
        combine_meis,
        display_path,
        find_camat_root,
        prepare_pages_for_combine,
        resolve_mei_inputs,
        resolve_repo_path,
    )
except ModuleNotFoundError:
    import setup_camat
    from camat import (
        combine_meis,
        display_path,
        find_camat_root,
        prepare_pages_for_combine,
        resolve_mei_inputs,
        resolve_repo_path,
    )

## 1. Point at your page files

`MEI_INPUTS` can be a folder of `*.mei` files, or a list of files. A folder is sorted by filename, so name pages so that order is the score order (for example `…_00126_…`, `…_00127_…`). You can also list files explicitly in the order they should appear.

`COMBINED_STEM` becomes `{COMBINED_STEM}_full.mei`. Leave it empty to name the file from the first page.

Review the plan, then set `RUN_COMBINE = True`.

In [ ]:
ROOT = find_camat_root()

# Folder of page files, or a list of .mei paths in score order.
MEI_INPUTS = ["test_corpus/buxtehude_pages"]
# Explicit list instead of a folder:
# MEI_INPUTS = [
#     "test_corpus/buxtehude_pages/bsb00023199_00126_facs_zones.mei",
#     "test_corpus/buxtehude_pages/bsb00023199_00127_facs_zones.mei",
# ]

# Combined file basename. Empty = use the first page filename.
COMBINED_STEM = "buxtehude_pages"

# Where to write the joined score (gitignored).
TARGET_DIR = "converted_mei/combine_tutorial"

# Writes stay off until this is True.
RUN_COMBINE = False

## 2. Review the plan

This cell only lists the files that would be joined. It does not write anything.

In [ ]:
target_dir = resolve_repo_path(TARGET_DIR, repo_root=ROOT)
page_files = resolve_mei_inputs(MEI_INPUTS, ROOT)
if not page_files:
    raise FileNotFoundError("No .mei files found in MEI_INPUTS")
if len(page_files) < 2:
    raise ValueError("Combining needs at least two page files.")

stem = COMBINED_STEM.strip() if COMBINED_STEM else page_files[0].stem
combined_path = target_dir / f"{stem}_full.mei"

print(f"Pages:      {len(page_files)}")
for path in page_files:
    print(f"  {display_path(path, repo_root=ROOT)}")
print(f"Write to:   {display_path(combined_path, repo_root=ROOT)}")
print(f"Run enabled: {RUN_COMBINE}")

## 3. Join the pages

With `RUN_COMBINE = True`, this cell writes one combined MEI under `TARGET_DIR`. Originals in `MEI_INPUTS` stay unchanged.

In [ ]:
if not RUN_COMBINE:
    print("Skipped. Review the plan, then set RUN_COMBINE = True.")
    combined_mei = None
else:
    import shutil
    import tempfile
    from pathlib import Path

    target_dir.mkdir(parents=True, exist_ok=True)
    prepare_dir = Path(tempfile.mkdtemp(prefix="camat_combine_"))
    try:
        prepared = prepare_pages_for_combine(page_files, prepare_dir)
        print(prepared.message)
        result = combine_meis(
            prepared.files,
            combined_path,
            renumber_measures=True,
            number_measure_zones=True,
            normalize_single_layer_numbers=True,
            keep_original_staff_names=True,
            staff_names_only_at_start=True,
        )
        combined_mei = result.path
        print(f"Wrote {display_path(result.path, repo_root=ROOT)}")
        print(f"Pages joined: {len(prepared.files)}")
        print(f"Duplicate xml:id values: {len(result.duplicate_ids)}")
        print(f"Renumbered measures: {result.renumbered_measures}")
        print(f"Inserted page scoreDefs: {result.inserted_page_score_defs}")
    finally:
        shutil.rmtree(prepare_dir, ignore_errors=True)

## 4. What was produced?

| File | Meaning |
| --- | --- |
| `{COMBINED_STEM}_full.mei` | the joined score |

**Next steps**

1. Open that file in [`mei_check_report.ipynb`](mei_check_report.ipynb) and write a CSV report.
2. Inspect notation and facsimile links in [`mei_facsimile_viewer.ipynb`](mei_facsimile_viewer.ipynb).
3. Correct remaining issues in [mei-friend](https://mei-friend.mdw.ac.at/), then re-check.

To join a different piece, change `MEI_INPUTS` and `COMBINED_STEM`, set `RUN_COMBINE` back to `False` while you review the plan, then enable it again.